In [1]:
import csv
import pandas as pd
import numpy as np
import re
from pathlib import Path

# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes VICON
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VICON_DIR = DATA_DIR / "VICON_CSV"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VICON_DIR =", VICON_DIR)
print("EXCEL_DIR =", EXCEL_DIR)
# --------------------------
# 1) Lecture CSV Vicon (ton format)
# --------------------------
def read_vicon_csv(csv_path: Path) -> pd.DataFrame:
    """
    Parse un CSV Vicon avec:
    L1: Trajectories
    L2: 100
    L3: noms marqueurs (avec vides)
    L4: Frame/Sub Frame puis X/Y/Z
    L5: unités (mm)
    L6+: données
    """
    with open(csv_path, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        header_lines = [next(reader) for _ in range(5)]

    marker_row = header_lines[2]   # ligne 3
    axis_row   = header_lines[3]   # ligne 4

    n = max(len(marker_row), len(axis_row))
    marker_row += [""] * (n - len(marker_row))
    axis_row   += [""] * (n - len(axis_row))

    # forward-fill des marqueurs (car Y/Z sont vides sous le nom de marqueur)
    filled = []
    last = ""
    for m in marker_row:
        m = (m or "").strip()
        if m == "":
            filled.append(last)
        else:
            last = m
            filled.append(last)

    # construit des noms plats: "<marker>_X", "<marker>_Y", "<marker>_Z"
    colnames = []
    for m, a in zip(filled, axis_row):
        m = (m or "").strip()
        a = (a or "").strip()

        if a in ["Frame", "Sub Frame"]:
            colnames.append(a)
        elif a in ["X", "Y", "Z"]:
            colnames.append(f"{m}_{a}")
        else:
            colnames.append(m if m else a)

    df = pd.read_csv(csv_path, skiprows=5, header=None, names=colnames, engine="python")
    df = df.dropna(axis=1, how="all")  # supprime colonnes vides trailing commas
    return df

# --------------------------
# 2) Trouver colonnes X/Y/Z d'un marker (préfixe variable ignoré)
# --------------------------
def find_xyz_cols(cols, token):
    """
    token: ex 'poignet_D', '2poignet_G', 'Tempe_D', etc.
    On match n'importe quel préfixe avant ':' : ex 'Patient 1:poignet_D_X'
    """
    pat = re.compile(rf"(?:^|:)\s*{re.escape(token)}_([XYZ])\b", re.IGNORECASE)
    found = {}
    for c in cols:
        c2 = c.replace(" ", "")
        m = pat.search(c2)
        if m:
            axis = m.group(1).upper()
            # si plusieurs colonnes match, on garde la plus courte (souvent la plus "propre")
            if axis not in found or len(c) < len(found[axis]):
                found[axis] = c
    return found.get("X"), found.get("Y"), found.get("Z")

# --------------------------
# 3) Quantité de mouvement d'un point 3D (mm)
# --------------------------
def motion_quantity_point(df, X, Y, Z):
    """
    QdM = somme des distances 3D frame->frame (mm)
    """
    arr = df[[X, Y, Z]].to_numpy(dtype=float)
    # diff entre frames
    d = np.diff(arr, axis=0)
    step = np.sqrt((d**2).sum(axis=1))
    # nettoie NaN/inf
    step = step[np.isfinite(step)]
    return float(step.sum()), int(step.size)

# --------------------------
# 4) Traitement d'un CSV
# --------------------------
def process_one_csv(csv_path: Path):
    df = read_vicon_csv(csv_path)
    cols = list(df.columns)

    out = {"csv": str(csv_path), "file": csv_path.name}

    # définition tokens (P1 vs P2)
    subjects = {
        "P1": {
            "WR_D": "poignet_D",
            "WR_G": "poignet_G",
            "TP_D": "Tempe_D",
            "TP_G": "Tempe_G",
        },
        "P2": {
            "WR_D": "2poignet_D",
            "WR_G": "2poignet_G",
            "TP_D": "2Tempe_D",
            "TP_G": "2Temps_G",
        }
    }

    for pid, tok in subjects.items():
        # ---- Poignets ----
        for side_key in ["WR_D", "WR_G"]:
            token = tok[side_key]
            X, Y, Z = find_xyz_cols(cols, token)
            if None in [X, Y, Z]:
                out[f"{pid}_{token}_error"] = "missing X/Y/Z"
            else:
                q, nsteps = motion_quantity_point(df, X, Y, Z)
                out[f"{pid}_{token}_QDM_mm"] = q
                out[f"{pid}_{token}_nsteps"] = nsteps

        # Total poignets (si dispo)
        wd = out.get(f"{pid}_{tok['WR_D']}_QDM_mm")
        wg = out.get(f"{pid}_{tok['WR_G']}_QDM_mm")
        if wd is not None and wg is not None:
            out[f"{pid}_QDM_WRISTS_mm"] = wd + wg

        # ---- Tête (tempes) ----
        for side_key in ["TP_D", "TP_G"]:
            token = tok[side_key]
            X, Y, Z = find_xyz_cols(cols, token)
            if None in [X, Y, Z]:
                out[f"{pid}_{token}_error"] = "missing X/Y/Z"
            else:
                q, nsteps = motion_quantity_point(df, X, Y, Z)
                out[f"{pid}_{token}_QDM_mm"] = q
                out[f"{pid}_{token}_nsteps"] = nsteps

        td = out.get(f"{pid}_{tok['TP_D']}_QDM_mm")
        tg = out.get(f"{pid}_{tok['TP_G']}_QDM_mm")
        if td is not None and tg is not None:
            out[f"{pid}_QDM_HEAD_mm"] = td + tg  # index tête = somme des 2 tempes

    return out

# --------------------------
# 5) Boucle dossier + export
# --------------------------
ROOT = VICON_DIR
csv_files = list(ROOT.rglob("*.csv"))
print("CSV found:", len(csv_files))

rows = []
for f in csv_files:
    rows.append(process_one_csv(f))

df_out = pd.DataFrame(rows)

out_path = EXCEL_DIR / "vicon_QDM_wrists_head_mm.xlsx"
df_out.to_excel(out_path, index=False)

print("✅ Saved:", out_path)

PROJECT_ROOT = /Users/matysprecloux/Desktop/SYNCOGEST
VICON_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/VICON_CSV
EXCEL_DIR = /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code
CSV found: 60
✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Excels_code/vicon_QDM_wrists_head_mm.xlsx
